# MedGemma Prostate Cancer Recommendation (PDQ PDF RAG)

This notebook builds a lightweight RAG pipeline over `PDQ.pdf` and uses retrieved chunks to ground treatment recommendations.

## Environment
```bash
conda activate /mnt/data9/conda/medgemma
jupyter lab
```

In [ ]:
# Optional installs if missing
# !pip install -U transformers accelerate torch pypdf sentence-transformers

In [1]:
import os
from pathlib import Path
import re
import numpy as np
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ['CUDA_VISIBLE_DEVICES'] = '1'

MODEL_ID = 'google/medgemma-1.5-4b-it'
PDF_PATH = Path('PDQ.pdf')
EMBED_MODEL_ID = 'sentence-transformers/all-MiniLM-L6-v2'

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    print('GPU0:', torch.cuda.get_device_name(0))

/mnt/data9/conda/medgemma/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
GPU count: 1
GPU0: NVIDIA H100 NVL


In [2]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f'{PDF_PATH} not found in current directory.')

reader = PdfReader(str(PDF_PATH))
pages = []
for i, p in enumerate(reader.pages, start=1):
    text = (p.extract_text() or '').strip()
    if text:
        pages.append({'page': i, 'text': text})

if not pages:
    raise ValueError('No extractable text found in PDQ.pdf.')

print('Loaded pages with text:', len(pages))
print('Total chars:', sum(len(p['text']) for p in pages))

Loaded pages with text: 152
Total chars: 384197


In [3]:
def chunk_page_text(page_num: int, text: str, chunk_chars: int = 1500, overlap: int = 200):
    chunks = []
    i = 0
    n = len(text)
    while i < n:
        j = min(i + chunk_chars, n)
        chunk_text = text[i:j].strip()
        if chunk_text:
            chunks.append({'page': page_num, 'text': chunk_text})
        if j == n:
            break
        i = max(j - overlap, i + 1)
    return chunks

chunks = []
for p in pages:
    chunks.extend(chunk_page_text(p['page'], p['text']))

print('Total chunks:', len(chunks))

Total chunks: 361


In [4]:
embedder = SentenceTransformer(EMBED_MODEL_ID)
chunk_texts = [c['text'] for c in chunks]
chunk_embeddings = embedder.encode(
    chunk_texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

print('Embedding shape:', chunk_embeddings.shape)

Batches: 100%|██████████| 12/12 [00:00<00:00, 23.35it/s]

Embedding shape: (361, 384)


In [5]:
def build_patient_query(patient_data: dict) -> str:
    return (
        f"Prostate cancer treatment guidance for: cT {patient_data['cT_stage']}, "
        f"cN {patient_data['cN_stage']}, cM {patient_data['cM_stage']}, "
        f"PSA {patient_data['psa_ng_ml']} ng/mL, ECOG {patient_data['performance_status_ecog']}, "
        f"age {patient_data['age_years']}, lesion count {patient_data['lesion_count']}, "
        f"prostate volume {patient_data['prostate_volume_ml']} mL, "
        f"estimated clinically significant disease probability "
        f"{patient_data['probability_clinically_significant_pc_percent']}%."
    )

def retrieve_pdq_chunks(query: str, top_k: int = 8):
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(chunk_embeddings, q_emb)
    top_idx = np.argsort(scores)[-top_k:][::-1]
    out = []
    for idx in top_idx:
        c = chunks[int(idx)]
        out.append({
            'page': c['page'],
            'score': float(scores[idx]),
            'text': c['text']
        })
    return out

patient = {
    'cT_stage': 'cT2a',
    'cN_stage': 'cN0',
    'cM_stage': 'cM0',
    'psa_ng_ml': 8.4,
    'performance_status_ecog': 0,
    'age_years': 66,
    'probability_clinically_significant_pc_percent': 62,
    'lesion_count': 2,
    'prostate_volume_ml': 48
}

query = build_patient_query(patient)
retrieved = retrieve_pdq_chunks(query, top_k=8)

for i, r in enumerate(retrieved, start=1):
    print(f"{i}. page={r['page']} score={r['score']:.4f}")

1. page=76 score=0.6649
2. page=118 score=0.6601
3. page=120 score=0.6590
4. page=37 score=0.6587
5. page=14 score=0.6572
6. page=150 score=0.6550
7. page=85 score=0.6460
8. page=15 score=0.6391


In [6]:
def build_prompt(patient_data: dict, retrieved_chunks: list[dict]) -> str:
    context_blocks = []
    for r in retrieved_chunks:
        context_blocks.append(f"[PDQ page {r['page']}]\n{r['text']}")
    pdq_context = '\n\n'.join(context_blocks)

    return f"""You are a specialist-facing prostate cancer decision-support assistant.

Use only the retrieved PDQ context for treatment claims.
If key variables are missing (for example histopathology/ISUP grade), explicitly state uncertainty and required next steps.
Cite page numbers for each major recommendation using format [PDQ p.X].

Patient data:
- cT stage: {patient_data['cT_stage']}
- cN stage: {patient_data['cN_stage']}
- cM stage: {patient_data['cM_stage']}
- PSA (ng/mL): {patient_data['psa_ng_ml']}
- Performance status (ECOG): {patient_data['performance_status_ecog']}
- Age (years): {patient_data['age_years']}
- Probability of clinically significant prostate cancer (%): {patient_data['probability_clinically_significant_pc_percent']}
- Lesion count: {patient_data['lesion_count']}
- Prostate volume (mL): {patient_data['prostate_volume_ml']}

Retrieved PDQ context:
{pdq_context}

Return EXACTLY:
1) Risk summary
2) Likely risk-group classification with rationale
3) PDQ-aligned treatment options
4) Preferred recommendation and why
5) Contraindications / caution points
6) Required additional tests/clarifications
7) Confidence (High/Medium/Low) + reason
8) Safety disclaimer

Do not provide definitive diagnosis. Keep concise specialist style."""

prompt = build_prompt(patient, retrieved)
print('Prompt chars:', len(prompt))

Prompt chars: 4058


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map={'': 0} if torch.cuda.is_available() else 'cpu'
)

messages = [{'role': 'user', 'content': prompt}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors='pt'
)

if torch.cuda.is_available():
    inputs = inputs.to(model.device)

with torch.no_grad():
    out_ids = model.generate(
        inputs,
        max_new_tokens=900,
        do_sample=False
    )

gen = out_ids[0][inputs.shape[-1]:]
response = tokenizer.decode(gen, skip_special_tokens=True)
print(response)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.28it/s]
Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 13] Permission denied: '/home/zohaib/.cache/huggingface/hub/models--google--medgemma-1.5-4b-it/.no_exist/e9792da5fb8ee651083d345ec4bce07c3c9f1641/custom_generate/generate.py'
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


1)  **Risk summary:** The patient has a 62% probability of clinically significant prostate cancer. The PSA level of 8.4 ng/mL is elevated. The Gleason score is not provided, which is a key factor in risk stratification. The lesion count is 2. The prostate volume is 48 mL. The patient is 66 years old with an ECOG score of 0, indicating good performance status. The cT stage is cT2a, cN stage is cN0, and cM stage is cM0, indicating localized disease without nodal or distant metastasis.

2)  **Likely risk-group classification with rationale:** Based on the provided information (PSA 8.4 ng/mL, Gleason score not provided, 2 lesions, cT2a, cN0, cM0), the patient likely falls into the intermediate risk category. The PSA level is elevated, suggesting a higher risk than low-risk, but the Gleason score is crucial for precise classification. The presence of 2 lesions and cT2a stage also contributes to the intermediate risk assessment.

3)  **PDQ-aligned treatment options:**
    *   **Active Survei

## Notes
- Retrieval is embedding-based over chunks from `PDQ.pdf` (RAG).
- You can tune `chunk_chars`, `overlap`, and `top_k` for recall vs. context size.
- Keep clinician review in the loop for any treatment decision.